# MediScope AI — Exploratory Data Analysis & Research Questions Notebook

**Course:** Mathematics for AI-II (Artificial Intelligence Programme)  
**Dataset:** New York State Department of Health — SPARCS De-Identified Hospital Inpatient Discharges (2021)  

This notebook demonstrates programmatic verification of the 2,101,588-row dataset, preprocessing audit, statistical hypothesis testing for RQ1–RQ5, and ML model benchmarking.

In [ ]:
import json
import pandas as pd
import numpy as np
from scipy import stats

from src.config import PROCESSED_PARQUET_PATH, CLOUD_SUMMARY_JSON_PATH, MODEL_METADATA_PATH
from src.eda import compute_core_kpis, summarize_diagnosis, summarize_severity, summarize_payment

df = pd.read_parquet(PROCESSED_PARQUET_PATH)
print(f"Cleaned dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(3)

In [ ]:
kpis = compute_core_kpis(df)
pd.Series(kpis)

## RQ1: Diagnosis Categories With Longest Average Inpatient Stay (`n >= 500`)

In [ ]:
diag_summary = summarize_diagnosis(df, min_discharges=500)
diag_summary.head(15)

## RQ3: Severity vs. Charges, Costs, and Length of Stay (+ Kruskal-Wallis Test)

In [ ]:
sev_summary = summarize_severity(df)
display(sev_summary)

groups = [
    g["total_charges"].to_numpy()
    for _, g in df[df["apr_severity_description"] != "Not Applicable"].groupby("apr_severity_description", observed=True)
]
h_stat, p_val = stats.kruskal(*groups)
print(f"Kruskal-Wallis H = {h_stat:,.2f}, p = {p_val:.3e}")

## RQ5: Primary Payment Typology Comparison

In [ ]:
summarize_payment(df)

## Machine Learning Benchmark Summary

In [ ]:
with open(MODEL_METADATA_PATH, "r", encoding="utf-8") as f:
    ml_meta = json.load(f)
pd.DataFrame(ml_meta["model_comparison"])